# Inventory Pickings: Landing to Bronze

This notebook calls the production Landing-to-Bronze helper. Only parser-corrupt rows are quarantined.


In [ ]:
from pathlib import Path
import os, sys
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
PROJECT_ROOT = next((p for p in _candidates if (p / "pipelines").is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p / "repo" / "product_analysis" for p in _candidates if (p / "repo" / "product_analysis" / "pipelines").is_dir()), None)
if PROJECT_ROOT is None: raise RuntimeError("Could not locate product_analysis project root")
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = PROJECT_ROOT.parent.parent / "data"


In [ ]:
from pipelines.landing_to_bronze_common import run_landing_to_bronze, start_landing_spark
SOURCE_FILE = Path(os.environ.get("SOURCE_FILE", DATA_ROOT / r"inventory/pickings/2026-10-07_23_17.csv"))
MANIFEST_FILE = os.environ.get("MANIFEST_FILE", str(DATA_ROOT / "source-manifest.json"))
SCHEMA_FILE = PROJECT_ROOT / "schemas" / "inventory_pickings_schema.json"
BATCH_ID = os.environ.get("BATCH_ID")
BRONZE_PATH = os.environ.get("BRONZE_PATH", "hdfs://localhost:9000/capstone/warehouse/bronze/inventory_pickings")
AUDIT_TABLE = os.environ.get("LANDING_AUDIT_TABLE", "datalake.operations.schema_drift")
if not SOURCE_FILE.is_file(): raise FileNotFoundError(f"Supply the inventory_pickings CSV with SOURCE_FILE: {SOURCE_FILE}")


In [ ]:
spark = start_landing_spark("inventory_pickings_landing_to_bronze")
try:
    ingestion_result = run_landing_to_bronze(spark=spark, dataset_name="inventory_pickings", dataset_key="inventory/pickings", source_file=str(SOURCE_FILE), bronze_path=BRONZE_PATH, schema_file=str(SCHEMA_FILE), batch_id=BATCH_ID, manifest_file=MANIFEST_FILE, audit_table=AUDIT_TABLE)
    ingestion_result
finally:
    spark.stop()
